# Annotation and ground truth

Runs **locally** (Jupyter on this laptop, see `ANNOTATING.md`) or on Colab. No GPU needed.

1. Each annotator sets `ANNOTATOR` and `SPLIT` and labels every item. Read `ANNOTATION_GUIDELINES.md` first.
2. After both annotators finish a split, run the *Agreement* section. A third team member fills `final_value` in `annotations/disagreements_{split}.csv`, then the section is run again.

Order: both label **dev** → check material kappa (≥ 0.6, else refine the guidelines and re-label dev) → both label **test**.

In [ ]:
# Setup: works locally and on Colab.
import os, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Yashas-BR/grounded-product-description.git"
    CODE_DIR = "/content/ghd"
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.isdir(CODE_DIR):
        !git -C {CODE_DIR} pull --ff-only
    else:
        !git clone {REPO_URL} {CODE_DIR}
    os.chdir(CODE_DIR)
    !pip install -q pyyaml krippendorff
    CONFIGS = []
else:
    # Local: this notebook lives in ghd/notebooks; work from the repo root.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    CONFIGS = ["configs/local.yaml"]
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from groundedgen import utils
cfg = utils.load_config(extra_paths=CONFIGS)
print("project_dir:", cfg["project_dir"])

## Label

Category and color are pre-filled from metadata — check them against the image. Material starts blank and must be chosen. Every *Save & Next* writes the CSV immediately, so you can stop at any time and resume later: re-running the cell continues at the first unlabelled item.

In [ ]:
ANNOTATOR = "b"   # "a" or "b" (configs/default.yaml: annotation.annotators)
SPLIT = "dev"     # label dev first, test only after the guidelines are frozen

from groundedgen import annotate
assert ANNOTATOR in cfg["annotation"]["annotators"], cfg["annotation"]["annotators"]
session = annotate.launch(cfg, SPLIT, ANNOTATOR)

## Agreement and gold

Writes `annotations/disagreements_{split}.csv`, `data/manifest_final.csv` and `data/ground_truth_report.json`. Safe to re-run: filled-in `final_value`s are kept.

In [ ]:

import json
from groundedgen import agreement

final, report = agreement.build_ground_truth(cfg)
for split, agr in report["agreement"].items():
    if agr is None:
        print(f"{split}: no annotations yet")
        continue
    print(f"\n{split}:")
    for attr in ("category", "color", "material"):
        a = agr[attr]
        print(f"  {attr:9s} kappa={a['kappa']:.3f}  raw={a['raw_agreement']:.3f}  n={a['n_items']}")
    if agr["material_kappa_below_threshold"]:
        print("  ! material kappa below 0.6: discuss disagreements, refine guidelines, re-label")
    print(f"  unresolved pairs: {len(report['unresolved'][split])}")
print(json.dumps(report["counts"], indent=2))
